# 02 - Train 4 CTGAN (UNSW-NB15, 4 phien ban dataset)

Moi dataset -> 1 CTGAN -> 1 file synthetic.
Datasets: `original`, `ros_10k`, `smote_3k`, `smote_10k`.

Dat 4 file `unsw_nb15_attacks_*.csv` vao `dataset/UNSW_NB15/oversampled/` truoc khi chay.

In [6]:
# !pip install ctgan
import time
from pathlib import Path

import pandas as pd
import torch
from ctgan import CTGAN

print("CUDA:", torch.cuda.is_available())

CUDA: False


In [7]:
# ---- CONFIG ----
ROOT = Path.cwd().parent                      # notebook nam trong notebooks/ -> ROOT = Synthguard-AI/
DATA_DIR = ROOT / "dataset" / "UNSW_NB15" / "oversampled"
OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(exist_ok=True)

DATASETS = ["original", "ros_10k", "smote_3k", "smote_10k"]
EPOCHS = 10          # model khong can perfect; chay thu EPOCHS=5 truoc de do thoi gian
BATCH_SIZE = 500     # CTGAN yeu cau batch_size chia het cho pac (mac dinh 10)
SEED = 42
SKIP_IF_DONE = True  # neu da co file .pt + synthetic csv thi bo qua dataset do

LABEL_CANDIDATES = ["attack_cat", "label", "Label", "attack_type"]

In [8]:
# ---- Xem nhanh 4 dataset ----
data = {}
for name in DATASETS:
    path = DATA_DIR / f"unsw_nb15_attacks_{name}.csv"
    df = pd.read_csv(path)
    data[name] = df
    print(f"{name:10s} shape={df.shape}")

label_col = next((c for c in LABEL_CANDIDATES if c in data["original"].columns), None)
print("\nLabel column:", label_col)
if label_col:
    summary = pd.DataFrame({n: d[label_col].value_counts() for n, d in data.items()}).fillna(0).astype(int)
    display(summary)

original   shape=(79446, 49)
ros_10k    shape=(123977, 49)
smote_3k   shape=(86003, 49)
smote_10k  shape=(123977, 49)

Label column: Label


,original,ros_10k,smote_3k,smote_10k
Label,,,,
1,79446,123977,86003,123977


In [9]:
obj_cols = data["original"].select_dtypes(exclude="number").columns.tolist()
print("Cot kieu chuoi:", obj_cols)
for c in obj_cols:
    display(pd.DataFrame({n: d[c].value_counts() for n, d in data.items()}).fillna(0).astype(int))

CONST_COLS = [c for c in data["original"].columns if data["original"][c].nunique() == 1]
print("Cot hang so:", CONST_COLS)
label_col = obj_cols[0]      # doi lai neu cot attack khong phai cot nay

Cot kieu chuoi: ['Attack']


,original,ros_10k,smote_3k,smote_10k
Attack,,,,
Analysis,981,10000,3000,10000
Backdoor,3106,10000,3106,10000
DoS,4300,10000,4300,10000
Exploits,32390,32390,32390,32390
Fuzzers,21587,21587,21587,21587
Generic,6087,10000,6087,10000
Reconnaissance,9533,10000,9533,10000
Shellcode,1350,10000,3000,10000
Worms,112,10000,3000,10000


Cot hang so: ['Label']


In [10]:
import numpy as np

def clean(df):
    df = df.copy()
    df = df.replace([np.inf, -np.inf], np.nan)
    num_cols = df.select_dtypes(include=[np.number]).columns
    obj_cols = df.select_dtypes(exclude=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    df[obj_cols] = df[obj_cols].fillna("missing")
    return df

timings = {}
for name in DATASETS:
    model_path = OUT_DIR / f"ctgan_unsw_nb15_{name}.pt"
    syn_path = OUT_DIR / f"synthetic_ctgan_unsw_nb15_{name}.csv"
    if SKIP_IF_DONE and model_path.exists() and syn_path.exists():
        print(f"[{name}] da co output, bo qua")
        continue

    full = clean(data[name])
    data[name] = full                                   # cell so sanh dung ban da lam sach
    df = full.drop(columns=CONST_COLS)                  # MOI: bo cot hang so khi train
    cat_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    if label_col not in cat_cols:
        cat_cols.append(label_col)

    print(f"\n[{name}] rows={len(df)}, cols={df.shape[1]}, categorical={cat_cols}")
    torch.manual_seed(SEED)
    t0 = time.time()
    try:
        model = CTGAN(epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=True, enable_gpu=torch.cuda.is_available())
    except TypeError:
        model = CTGAN(epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=True, cuda=torch.cuda.is_available())
    model.fit(df, discrete_columns=cat_cols)

    model.save(str(model_path))
    syn = model.sample(len(df))
    for c in CONST_COLS:                                # MOI: gan lai cot hang so
        syn[c] = full[c].iloc[0]
    syn = syn[full.columns]                             # MOI: giu dung thu tu cot nhu file goc
    syn.to_csv(syn_path, index=False)

    timings[name] = (time.time() - t0) / 60
    print(f"[{name}] xong sau {timings[name]:.1f} phut -> {syn_path.name}")

print("\nThoi gian (phut):", {k: round(v, 1) for k, v in timings.items()})


[original] rows=79446, cols=48, categorical=['Attack']


Gen. (-00.14) | Discrim. (+00.67): 100%|██████████| 10/10 [04:37<00:00, 27.74s/it]


[original] xong sau 11.2 phut -> synthetic_ctgan_unsw_nb15_original.csv

[ros_10k] rows=123977, cols=48, categorical=['Attack']


Gen. (+00.93) | Discrim. (-00.35): 100%|██████████| 10/10 [05:27<00:00, 32.76s/it]


[ros_10k] xong sau 15.0 phut -> synthetic_ctgan_unsw_nb15_ros_10k.csv

[smote_3k] rows=86003, cols=48, categorical=['Attack']


Gen. (-00.67) | Discrim. (-00.03): 100%|██████████| 10/10 [03:48<00:00, 22.83s/it]


[smote_3k] xong sau 11.2 phut -> synthetic_ctgan_unsw_nb15_smote_3k.csv

[smote_10k] rows=123977, cols=48, categorical=['Attack']


Gen. (-00.96) | Discrim. (-00.19): 100%|██████████| 10/10 [05:20<00:00, 32.07s/it]


[smote_10k] xong sau 15.9 phut -> synthetic_ctgan_unsw_nb15_smote_10k.csv

Thoi gian (phut): {'original': 11.2, 'ros_10k': 15.0, 'smote_3k': 11.2, 'smote_10k': 15.9}


In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "dataset" / "UNSW_NB15" / "oversampled"
OUT_DIR = ROOT / "outputs"
DATASETS = ["original", "ros_10k", "smote_3k", "smote_10k"]
label_col = "Attack"

rows = {}
for name in DATASETS:
    syn_path = OUT_DIR / f"synthetic_ctgan_unsw_nb15_{name}.csv"
    if not syn_path.exists():
        print(f"[{name}] chua co file synthetic")
        continue
    real = pd.read_csv(DATA_DIR / f"unsw_nb15_attacks_{name}.csv")
    syn = pd.read_csv(syn_path)
    print(f"[{name}] real={real.shape}, syn={syn.shape}")
    rows[f"{name}_real"] = real[label_col].value_counts(normalize=True)
    rows[f"{name}_syn"] = syn[label_col].value_counts(normalize=True)

display((pd.DataFrame(rows).fillna(0) * 100).round(2))

[original] real=(79446, 49), syn=(79446, 49)
[ros_10k] real=(123977, 49), syn=(123977, 49)
[smote_3k] real=(86003, 49), syn=(86003, 49)
[smote_10k] real=(123977, 49), syn=(123977, 49)


,original_real,original_syn,ros_10k_real,ros_10k_syn,smote_3k_real,smote_3k_syn,smote_10k_real,smote_10k_syn
Attack,,,,,,,,
Analysis,1.23,9.36,8.07,10.93,3.49,10.24,8.07,10.85
Backdoor,3.91,10.99,8.07,10.90,3.61,10.35,8.07,10.85
DoS,5.41,11.38,8.07,10.73,5.00,10.48,8.07,10.85
Exploits,40.77,14.11,26.13,12.23,37.66,13.12,26.13,12.21
Fuzzers,27.17,13.77,17.41,11.62,25.10,12.73,17.41,11.80
Generic,7.66,11.75,8.07,10.91,7.08,10.97,8.07,10.89
Reconnaissance,12.00,12.42,8.07,10.81,11.08,11.64,8.07,10.91
Shellcode,1.70,9.92,8.07,10.98,3.49,10.17,8.07,10.80
Worms,0.14,6.28,8.07,10.88,3.49,10.30,8.07,10.85
